## Setup

In [5]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

from gensim.corpora import Dictionary
from gensim.models import word2vec

from sklearn.manifold import TSNE as tsne

OUTPUT_DIR = 'tables'

In [6]:
import warnings; warnings.filterwarnings('ignore', category=FutureWarning)
import gensim; gensim.__version__

'4.4.0'

In [7]:
OHCO = ['film_name', 'chunk_num', 'sent_num', 'token_num']
bags = dict(
    SENTS = OHCO[:3],
    CHUNKS = OHCO[:2],
    FILMS = OHCO[:1]
)

# Import tables
LIB = pd.read_csv('tables/LIB.csv').set_index('film_name')
CORPUS = pd.read_csv('tables/CORPUS.csv').set_index(OHCO)

## Convert to Gensim

In [8]:
docs = CORPUS.groupby(bags['CHUNKS']).term_str.apply(list).tolist()

In [9]:
# inspect first 5 documents of Gensim corpus
for i in range(5):
    print(f"Doc {i}:", docs[i])

Doc 0: ['ah']
Doc 1: ['wah', 'it', 's', 'a', 'gas', 'bomb', 'it', 's', 'an', 'attack']
Doc 2: ['wah']
Doc 3: ['ah', 'wah', 'hey']
Doc 4: ['hold', 'them', 'there', 'you', 'get', 'down', 'on', 'the', 'floor']


In [10]:
dictionary = Dictionary(docs)

## Generate Word Embeddings

In [11]:
w2v_params = dict(
    window = 10,
    vector_size = 100,
    sg = 1, # add skip-gram to train on word-context pair individually, given limited data
    min_count = 20,
    workers = 1,
    seed = 42
)

In [12]:
model = word2vec.Word2Vec(docs, **w2v_params)
model.wv.vectors

array([[-0.1469774 ,  0.02532332, -0.01587839, ..., -0.00138017,
        -0.07543324, -0.11003441],
       [ 0.15915978,  0.00303971,  0.14199327, ..., -0.03821651,
        -0.10147785, -0.09974402],
       [-0.09574553,  0.17313518,  0.18212691, ...,  0.04824068,
        -0.14449681, -0.29993716],
       ...,
       [-0.05647562,  0.11639901, -0.00235493, ...,  0.01665139,
        -0.07901005, -0.13590786],
       [-0.0424082 ,  0.07950125,  0.0063124 , ..., -0.02395798,
        -0.07506037, -0.10761096],
       [-0.0618719 ,  0.08860315,  0.00593048, ..., -0.00476142,
        -0.07336204, -0.11415197]], shape=(577, 100), dtype=float32)

In [13]:
VOCAB_W2V = pd.DataFrame(model.wv.vectors, index=model.wv.index_to_key)
VOCAB_W2V.index.name = 'term_str'
VOCAB_W2V

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
term_str,,,,,,,,,,,,,,,,,,,,,
the,-0.146977,0.025323,-0.015878,0.154392,-0.304920,0.344824,-0.083911,0.243857,-0.206707,-0.101089,...,-0.019569,0.313504,0.096867,0.098971,0.101148,-0.012143,-0.004169,-0.001380,-0.075433,-0.110034
you,0.159160,0.003040,0.141993,-0.083904,-0.000782,0.165063,0.133079,0.106512,-0.096537,0.149314,...,-0.061997,0.087598,0.168231,-0.268038,0.105066,-0.071198,0.072113,-0.038217,-0.101478,-0.099744
i,-0.095746,0.173135,0.182127,0.101254,0.013877,-0.005864,0.047570,0.244175,-0.037220,0.226866,...,-0.095547,0.143045,0.314396,-0.415740,0.066235,0.026825,0.065082,0.048241,-0.144497,-0.299937
s,-0.182741,0.228977,-0.110764,0.194951,-0.121366,-0.126499,-0.232623,-0.083822,-0.206088,0.223280,...,0.040597,0.012933,0.068314,0.039012,0.124493,0.280368,-0.000961,-0.235532,-0.328034,-0.042798
to,0.148520,0.144695,-0.010893,0.145829,-0.254088,0.124180,0.015815,0.094733,-0.086298,-0.083857,...,0.022577,0.180355,0.237381,-0.193607,0.007439,0.058010,0.032182,0.037079,0.068112,-0.193118
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
friends,-0.045363,0.069975,0.011364,0.100147,-0.252640,0.200875,-0.037881,0.149812,-0.178572,0.056010,...,-0.166048,0.114741,0.070806,-0.069585,0.023649,-0.001649,0.038182,-0.010530,-0.077605,-0.130001
shut,-0.019376,0.060844,0.003100,0.083031,-0.214571,0.108156,-0.066533,0.099885,-0.182831,0.065906,...,-0.121350,0.145319,0.095373,-0.075656,-0.005192,0.047828,0.049020,-0.021938,-0.042629,-0.140792
clan,-0.056476,0.116399,-0.002355,0.100075,-0.316138,0.266258,-0.029846,0.218051,-0.177996,0.015636,...,-0.169714,0.125731,0.083609,-0.019742,0.024313,-0.019711,0.061084,0.016651,-0.079010,-0.135908


## Visualize with tSNE

In [14]:
PP = 40 # Try 1, 100, etc.

tsne_engine = tsne(
    perplexity=PP, 
    n_components=2, 
    init='random', 
    max_iter=2500, 
    random_state=23
)
TSNE = pd.DataFrame(
    tsne_engine.fit_transform(VOCAB_W2V), 
    columns=['x','y'], 
    index=VOCAB_W2V.index)
TSNE

,x,y
term_str,,
the,23.088427,-9.094011
you,-13.645277,-7.679845
i,-18.898586,-2.458489
s,-0.755515,-7.988135
to,-7.716119,12.432200
...,...,...
friends,11.983424,-4.755201
shut,-1.783963,7.825482
clan,18.477646,-4.724789


In [15]:
px.scatter(TSNE.reset_index(), 'x', 'y', 
        text='term_str', 
        hover_name='term_str',  
        height=1000,
        width=1200)\
    .update_traces(
        mode='markers+text', 
        textfont=dict(color='black', size=14, family='Arial'),
        textposition='top center')

## Save Tables

In [16]:
VOCAB_W2V.to_csv(f'{OUTPUT_DIR}/w2v.csv')